In [ ]:
import nfl_data_py as nfl
import tensorflow as tf
from sklearn.model_selection import train_test_split
import numpy as np
import pandas as pd

nfl_df = nfl.import_seasonal_data([2024], 'REG')
weekly = nfl.import_weekly_data([2024])
weekly.replace('Missing value',value=0, inplace=True)
weekly_y = weekly['fantasy_points_ppr']
weekly_x = weekly.drop(['fantasy_points','fantasy_points_ppr', 'headshot_url'], axis=1)
weekly_x['current_season'] = np.where(weekly_x['season'] == '2025', 1 , 0)
train_x, test_x, train_y, test_y = train_test_split(weekly_x,weekly_y,train_size=0.2)

weekly_x.head()

Downcasting floats.


,player_id,player_name,player_display_name,position,position_group,recent_team,season,week,season_type,opponent_team,...,receiving_yards_after_catch,receiving_first_downs,receiving_epa,receiving_2pt_conversions,racr,target_share,air_yards_share,wopr,special_teams_tds,current_season
0,00-0023459,A.Rodgers,Aaron Rodgers,QB,QB,NYJ,2024,1,REG,SF,...,0.0,0.0,NaN,0,NaN,NaN,NaN,NaN,0.0,0
1,00-0023459,A.Rodgers,Aaron Rodgers,QB,QB,NYJ,2024,2,REG,TEN,...,0.0,0.0,NaN,0,NaN,NaN,NaN,NaN,0.0,0
2,00-0023459,A.Rodgers,Aaron Rodgers,QB,QB,NYJ,2024,3,REG,NE,...,0.0,0.0,NaN,0,NaN,NaN,NaN,NaN,0.0,0
3,00-0023459,A.Rodgers,Aaron Rodgers,QB,QB,NYJ,2024,4,REG,DEN,...,0.0,0.0,NaN,0,NaN,NaN,NaN,NaN,0.0,0
4,00-0023459,A.Rodgers,Aaron Rodgers,QB,QB,NYJ,2024,5,REG,MIN,...,0.0,0.0,NaN,0,NaN,NaN,NaN,NaN,0.0,0


In [42]:
# Exploring New Dataset that has 2025 stats, but it lacks column names
import nflreadpy as nfl2
stats = pd.DataFrame(nfl2.load_player_stats([2024, 2025]))
stats.to_csv('stats.csv',index=False)


In [41]:
normalization_layer = tf.keras.layers.Normalization()
hidden_layer1 = tf.keras.layers.Dense(30, activation='relu')
hidden_layer2 = tf.keras.layers.Dense(30, activation='relu')
concatenate_layer = tf.keras.layers.Concatenate()
output_layer = tf.keras.layers.Dense(1)

nn_input = tf.keras.Input(shape=train_x.shape[1:])
normalization = normalization_layer(nn_input)
hidden_layer = hidden_layer1(normalization)
hidden_layer = hidden_layer2(hidden_layer)
concatenate = concatenate_layer([normalization,hidden_layer])
output = output_layer(concatenate)

model = tf.keras.Model(inputs=nn_input, outputs=output)

model.compile(optimizer='adam', loss='mse')